In [1]:
import os
import glob

print("GPU files:")
print(os.listdir("/kaggle/input"))

print("\nAttached dataset files:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files[:10]:
        print(os.path.join(root, f))

GPU files:
['datasets']

Attached dataset files:
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/metadata/part1_train_ids.json
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/metadata/part1_manifest.csv
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/metadata/part1_val_ids.json
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00084_11.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00708_33.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00341_23.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00757_10.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00345_01.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00331_00.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/images/00896_30.npy
/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-

In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1"
)

IMAGE_DIR = DATA_ROOT / "images"
MASK_DIR = DATA_ROOT / "masks"
META_DIR = DATA_ROOT / "metadata"

# ============================================================
# GPU CHECK
# ============================================================

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA/GPU is not available. Enable GPU in Kaggle.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

# ============================================================
# DATASET CHECK
# ============================================================

print("\n" + "=" * 70)
print("DATASET CHECK")
print("=" * 70)

train_ids = json.load(open(META_DIR / "part1_train_ids.json"))
val_ids = json.load(open(META_DIR / "part1_val_ids.json"))

manifest = pd.read_csv(META_DIR / "part1_manifest.csv")

print("Train original images:", len(train_ids))
print("Val original images:  ", len(val_ids))
print("Manifest rows:        ", len(manifest))

train_rows = (manifest["split"] == "train").sum()
val_rows = (manifest["split"] == "val").sum()

print("Train patches:", train_rows)
print("Val patches:  ", val_rows)

# ============================================================
# SAMPLE CHECK
# ============================================================

sample_image = next(IMAGE_DIR.glob("*.npy"))
sample_mask = MASK_DIR / sample_image.name

x = np.load(sample_image)
y = np.load(sample_mask)

print("\nSample:")
print("Image:", sample_image.name)
print("Image shape:", x.shape)
print("Image dtype:", x.dtype)
print("Mask shape:", y.shape)
print("Mask dtype:", y.dtype)
print("Mask unique values:", np.unique(y))

assert x.shape == (2, 512, 512)
assert y.shape == (512, 512)
assert x.dtype == np.float32
assert y.dtype == np.uint8

print("\n✓ Dataset verification passed.")

# ============================================================
# SAMPLE VALUE STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("SAR VALUE CHECK")
print("=" * 70)

print("VV min/max:", float(x[0].min()), float(x[0].max()))
print("VH min/max:", float(x[1].min()), float(x[1].max()))
print("VV mean/std:", float(x[0].mean()), float(x[0].std()))
print("VH mean/std:", float(x[1].mean()), float(x[1].std()))

# ============================================================
# CUDA BENCHMARK
# ============================================================

print("\n" + "=" * 70)
print("GPU MEMORY TEST")
print("=" * 70)

device = torch.device("cuda")

# Test several batch sizes.
for batch_size in [4, 8, 16]:
    try:
        torch.cuda.empty_cache()

        test = torch.randn(
            batch_size, 2, 512, 512,
            device=device,
            dtype=torch.float32
        )

        # Basic convolution test
        layer = torch.nn.Conv2d(2, 32, kernel_size=3, padding=1).to(device)

        torch.cuda.synchronize()
        start = time.time()

        for _ in range(10):
            out = layer(test)

        torch.cuda.synchronize()
        elapsed = time.time() - start

        del test, layer, out
        torch.cuda.empty_cache()

        print(
            f"Batch {batch_size:2d}: "
            f"PASS | {elapsed/10*1000:.1f} ms/batch"
        )

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"Batch {batch_size:2d}: OOM")
            torch.cuda.empty_cache()
        else:
            raise

print("\n✓ GPU benchmark complete.")
print("=" * 70)

GPU CHECK
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

DATASET CHECK
Train original images: 960
Val original images:   240
Manifest rows:         19200
Train patches: 15360
Val patches:   3840

Sample:
Image: 00084_11.npy
Image shape: (2, 512, 512)
Image dtype: float32
Mask shape: (512, 512)
Mask dtype: uint8
Mask unique values: [0]

✓ Dataset verification passed.

SAR VALUE CHECK
VV min/max: -37.78077697753906 -22.096210479736328
VH min/max: -23.11302947998047 -11.712955474853516
VV mean/std: -29.129947662353516 1.7189286947250366
VH mean/std: -17.631053924560547 1.2567452192306519

GPU MEMORY TEST
Batch  4: PASS | 66.9 ms/batch
Batch  8: PASS | 4.7 ms/batch
Batch 16: PASS | 9.0 ms/batch

✓ GPU benchmark complete.


In [3]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

GPU count: 2
0 Tesla T4
1 Tesla T4


In [6]:
%%writefile /kaggle/working/benchmark_ddp.py

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader, DistributedSampler


# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1"
)

BATCH_SIZE = 4       # PER GPU -> effective batch size = 8
NUM_WORKERS = 2      # PER GPU


# ============================================================
# DATASET
# ============================================================

manifest = pd.read_csv(
    DATA_ROOT / "metadata" / "part1_manifest.csv"
)


class SARDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = np.load(
            DATA_ROOT / row["image"]
        ).astype(np.float32)

        mask = np.load(
            DATA_ROOT / row["mask"]
        ).astype(np.float32)

        image = torch.from_numpy(image.copy())

        mask = torch.from_numpy(
            mask.copy()
        ).unsqueeze(0)

        mask = (mask > 0).float()

        return image, mask


train_ds = SARDataset(
    manifest[manifest["split"] == "train"]
)

val_ds = SARDataset(
    manifest[manifest["split"] == "val"]
)


# ============================================================
# U-NET
# ============================================================

class ConvBlock(nn.Module):

    def __init__(self, in_ch, out_ch):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.e1 = ConvBlock(2, 32)
        self.e2 = ConvBlock(32, 64)
        self.e3 = ConvBlock(64, 128)
        self.e4 = ConvBlock(128, 256)

        self.b = ConvBlock(256, 512)

        self.u4 = nn.ConvTranspose2d(
            512, 256, 2, stride=2
        )

        self.d4 = ConvBlock(512, 256)

        self.u3 = nn.ConvTranspose2d(
            256, 128, 2, stride=2
        )

        self.d3 = ConvBlock(256, 128)

        self.u2 = nn.ConvTranspose2d(
            128, 64, 2, stride=2
        )

        self.d2 = ConvBlock(128, 64)

        self.u1 = nn.ConvTranspose2d(
            64, 32, 2, stride=2
        )

        self.d1 = ConvBlock(64, 32)

        self.out = nn.Conv2d(
            32, 1, 1
        )

    def forward(self, x):

        e1 = self.e1(x)

        e2 = self.e2(
            self.pool(e1)
        )

        e3 = self.e3(
            self.pool(e2)
        )

        e4 = self.e4(
            self.pool(e3)
        )

        b = self.b(
            self.pool(e4)
        )

        x = self.u4(b)

        x = self.d4(
            torch.cat([x, e4], dim=1)
        )

        x = self.u3(x)

        x = self.d3(
            torch.cat([x, e3], dim=1)
        )

        x = self.u2(x)

        x = self.d2(
            torch.cat([x, e2], dim=1)
        )

        x = self.u1(x)

        x = self.d1(
            torch.cat([x, e1], dim=1)
        )

        return self.out(x)


# ============================================================
# LOSS
# ============================================================

def loss_fn(logits, target):

    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        target
    )

    prob = torch.sigmoid(logits)

    intersection = (
        prob * target
    ).sum(dim=(1, 2, 3))

    dice = (
        2 * intersection + 1
    ) / (
        prob.sum(dim=(1, 2, 3))
        +
        target.sum(dim=(1, 2, 3))
        + 1
    )

    return bce + (1 - dice.mean())


# ============================================================
# METRICS
# ============================================================

@torch.no_grad()
def get_metrics(logits, target):

    pred = (
        torch.sigmoid(logits) > 0.5
    ).float()

    intersection = (
        pred * target
    ).sum()

    dice = (
        2 * intersection + 1
    ) / (
        pred.sum()
        + target.sum()
        + 1
    )

    union = (
        pred.sum()
        + target.sum()
        - intersection
    )

    iou = (
        intersection + 1
    ) / (union + 1)

    return dice, iou


# ============================================================
# TRAINING
# ============================================================

def main():

    # --------------------------------------------------------
    # DDP
    # --------------------------------------------------------

    dist.init_process_group(
        backend="nccl"
    )

    rank = dist.get_rank()

    local_rank = int(
        os.environ["LOCAL_RANK"]
    )

    torch.cuda.set_device(local_rank)

    device = torch.device(
        f"cuda:{local_rank}"
    )

    # --------------------------------------------------------
    # Print GPU information
    # --------------------------------------------------------

    if rank == 0:

        print("=" * 70)
        print("REAL 2-GPU BENCHMARK")
        print("=" * 70)

        print(
            "GPU count:",
            torch.cuda.device_count()
        )

        for i in range(
            torch.cuda.device_count()
        ):
            print(
                f"GPU {i}:",
                torch.cuda.get_device_name(i)
            )

        print()

    # --------------------------------------------------------
    # Samplers
    # --------------------------------------------------------

    train_sampler = DistributedSampler(
        train_ds,
        shuffle=True,
        seed=42
    )

    val_sampler = DistributedSampler(
        val_ds,
        shuffle=False
    )

    # --------------------------------------------------------
    # DataLoaders
    # --------------------------------------------------------

    train_loader = DataLoader(

        train_ds,

        batch_size=BATCH_SIZE,

        sampler=train_sampler,

        num_workers=NUM_WORKERS,

        pin_memory=True,

        persistent_workers=True
    )

    val_loader = DataLoader(

        val_ds,

        batch_size=BATCH_SIZE,

        sampler=val_sampler,

        num_workers=NUM_WORKERS,

        pin_memory=True,

        persistent_workers=True
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = UNet().to(device)

    model = nn.parallel.DistributedDataParallel(
        model,
        device_ids=[local_rank]
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    scaler = torch.amp.GradScaler(
        "cuda"
    )

    # --------------------------------------------------------
    # Start benchmark
    # --------------------------------------------------------

    dist.barrier()

    start_time = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_sampler.set_epoch(0)

    total_batches = len(train_loader)

    running_loss = 0.0

    if rank == 0:

        print(
            f"Train patches: {len(train_ds):,}"
        )

        print(
            f"Validation patches: {len(val_ds):,}"
        )

        print(
            f"Batch/GPU: {BATCH_SIZE}"
        )

        print(
            f"Effective batch: "
            f"{BATCH_SIZE * 2}"
        )

        print()

    for batch_idx, (x, y) in enumerate(
        train_loader
    ):

        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            pred = model(x)

            loss = loss_fn(
                pred,
                y
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += loss.item()

        # ----------------------------------------------------
        # LIVE PROGRESS
        # ----------------------------------------------------

        if rank == 0:

            done = batch_idx + 1

            elapsed = (
                time.time()
                - start_time
            )

            rate = done / max(
                elapsed,
                0.001
            )

            remaining = (
                total_batches
                - done
            )

            eta = (
                remaining / max(
                    rate,
                    1e-9
                )
            )

            pct = (
                done
                / total_batches
                * 100
            )

            bar_len = 30

            filled = int(
                bar_len
                * done
                / total_batches
            )

            bar = (
                "█" * filled
                +
                "░" * (
                    bar_len - filled
                )
            )

            print(
                f"\rTrain "
                f"[{bar}] "
                f"{pct:6.2f}% | "
                f"{done}/{total_batches} | "
                f"loss {loss.item():.4f} | "
                f"ETA {eta/60:.1f} min",
                end="",
                flush=True
            )

    if rank == 0:
        print()

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    dice_sum = 0.0
    iou_sum = 0.0
    sample_count = 0

    val_batches = len(
        val_loader
    )

    with torch.no_grad():

        for batch_idx, (x, y) in enumerate(
            val_loader
        ):

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                pred = model(x)

            dice, iou = get_metrics(
                pred,
                y
            )

            n = x.size(0)

            dice_sum += (
                dice.item() * n
            )

            iou_sum += (
                iou.item() * n
            )

            sample_count += n

            if rank == 0:

                done = batch_idx + 1

                pct = (
                    done
                    / val_batches
                    * 100
                )

                bar_len = 30

                filled = int(
                    bar_len
                    * done
                    / val_batches
                )

                bar = (
                    "█" * filled
                    +
                    "░" * (
                        bar_len - filled
                    )
                )

                print(
                    f"\rVal   "
                    f"[{bar}] "
                    f"{pct:6.2f}% | "
                    f"{done}/{val_batches}",
                    end="",
                    flush=True
                )

    if rank == 0:
        print()

    # --------------------------------------------------------
    # Synchronize
    # --------------------------------------------------------

    torch.cuda.synchronize()

    dist.barrier()

    epoch_time = (
        time.time()
        - start_time
    )

    # --------------------------------------------------------
    # Combine metrics
    # --------------------------------------------------------

    stats = torch.tensor(
        [
            dice_sum,
            iou_sum,
            sample_count
        ],
        dtype=torch.float64,
        device=device
    )

    dist.all_reduce(
        stats,
        op=dist.ReduceOp.SUM
    )

    if rank == 0:

        dice = (
            stats[0].item()
            /
            stats[2].item()
        )

        iou = (
            stats[1].item()
            /
            stats[2].item()
        )

        print()
        print("=" * 70)
        print("BENCHMARK COMPLETE")
        print("=" * 70)

        print(
            f"Epoch time: "
            f"{epoch_time/60:.2f} minutes"
        )

        print(
            f"25 epochs: "
            f"{epoch_time*25/3600:.2f} hours"
        )

        print(
            f"50 epochs: "
            f"{epoch_time*50/3600:.2f} hours"
        )

        print()

        print(
            f"Validation Dice: "
            f"{dice:.4f}"
        )

        print(
            f"Validation IoU: "
            f"{iou:.4f}"
        )

        print("=" * 70)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/benchmark_ddp.py


In [7]:
!MASTER_ADDR=127.0.0.1 MASTER_PORT=29501 torchrun \
    --standalone \
    --nproc_per_node=2 \
    /kaggle/working/benchmark_ddp.py

W0916 17:07:46.490000 137 torch/distributed/run.py:852] 
W0916 17:07:46.490000 137 torch/distributed/run.py:852] *****************************************
W0916 17:07:46.490000 137 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0916 17:07:46.490000 137 torch/distributed/run.py:852] *****************************************
[W916 17:07:46.859342170 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 17:07:48.972583624 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 17:07:48.980220377 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
REAL 2-GPU BENCHMARK
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWar

In [10]:
import os

ROOT = "/kaggle/working"

# Delete empty files
deleted_files = 0
for root, dirs, files in os.walk(ROOT, topdown=False):
    for f in files:
        path = os.path.join(root, f)
        if os.path.isfile(path) and os.path.getsize(path) == 0:
            os.remove(path)
            deleted_files += 1

# Delete empty directories
deleted_dirs = 0
for root, dirs, files in os.walk(ROOT, topdown=False):
    for d in dirs:
        path = os.path.join(root, d)
        if os.path.isdir(path) and not os.listdir(path):
            os.rmdir(path)
            deleted_dirs += 1

print(f"Empty files deleted: {deleted_files}")
print(f"Empty directories deleted: {deleted_dirs}")
print("Done.")

Empty files deleted: 0
Empty directories deleted: 1
Done.


In [18]:
# ============================================================
# CELL 1 — FINAL ROBUST 2-GPU TRAINING SCRIPT
# ============================================================

from pathlib import Path
import os

WORK = "/kaggle/working/sar_oil_spill_robust"
BASE = "/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1"

Path(WORK).mkdir(parents=True, exist_ok=True)
Path(f"{WORK}/checkpoints").mkdir(parents=True, exist_ok=True)

script = r'''
import os
import json
import time
import random
import numpy as np

import torch
import torch.nn as nn
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP


# ============================================================
# CONFIG
# ============================================================

BASE = "/kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1"

WORK = "/kaggle/working/sar_oil_spill_robust"

CKPT_DIR = f"{WORK}/checkpoints"

MAX_EPOCHS = 100

# Original known-good model.
BEST_MODEL = (
    "/kaggle/working/sar_oil_training/"
    "checkpoints/best_model.pt"
)

# Conservative learning rate.
LR = 1e-4

WEIGHT_DECAY = 1e-4

BATCH_SIZE_PER_GPU = 4

NUM_WORKERS = 2

GRAD_CLIP = 1.0

MIN_DELTA = 0.001

CHECK_EVERY = 10

PATIENCE_BLOCKS = 2

SEED = 42


# ============================================================
# DDP INITIALIZATION
# ============================================================

local_rank = int(
    os.environ["LOCAL_RANK"]
)

rank = int(
    os.environ["RANK"]
)

world_size = int(
    os.environ["WORLD_SIZE"]
)

torch.cuda.set_device(
    local_rank
)

device = torch.device(
    "cuda",
    local_rank
)

dist.init_process_group(
    backend="nccl",
    init_method="env://"
)


# ============================================================
# SEED
# ============================================================

def seed_everything(seed):

    random.seed(
        seed + rank
    )

    np.random.seed(
        seed + rank
    )

    torch.manual_seed(
        seed + rank
    )

    torch.cuda.manual_seed_all(
        seed + rank
    )


seed_everything(SEED)


# ============================================================
# ORIGINAL DATASET
# ============================================================

class PatchDataset(Dataset):

    def __init__(
        self,
        manifest,
        split
    ):

        self.rows = []

        with open(
            manifest,
            "r"
        ) as f:

            header = (
                f.readline()
                .strip()
                .split(",")
            )

            for line in f:

                vals = (
                    line
                    .strip()
                    .split(",")
                )

                row = dict(
                    zip(
                        header,
                        vals
                    )
                )

                if row["split"] == split:

                    self.rows.append(
                        row
                    )


    def __len__(self):

        return len(
            self.rows
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.rows[idx]

        image = np.load(
            os.path.join(
                BASE,
                row["image"]
            ),
            mmap_mode="r"
        )

        mask = np.load(
            os.path.join(
                BASE,
                row["mask"]
            ),
            mmap_mode="r"
        )

        image = np.asarray(
            image,
            dtype=np.float32
        )

        mask = np.asarray(
            mask,
            dtype=np.float32
        )

        return (
            torch.from_numpy(
                image.copy()
            ),
            torch.from_numpy(
                mask.copy()
            ).unsqueeze(0)
        )


manifest = (
    f"{BASE}/metadata/"
    "part1_manifest.csv"
)


raw_train_ds = PatchDataset(
    manifest,
    "train"
)


raw_val_ds = PatchDataset(
    manifest,
    "val"
)


# ============================================================
# CALCULATE TRAINING STATISTICS
#
# IMPORTANT:
# Statistics are accumulated on CPU.
# Only the final values are moved to GPU.
# ============================================================

if rank == 0:

    print()
    print("=" * 70)
    print("CALCULATING TRAINING-SET NORMALIZATION")
    print("=" * 70)


local_sum = np.zeros(
    2,
    dtype=np.float64
)

local_sum_sq = np.zeros(
    2,
    dtype=np.float64
)

local_count = 0


for idx in range(
    rank,
    len(raw_train_ds),
    world_size
):

    image, _ = raw_train_ds[idx]

    image = image.numpy()

    if not np.isfinite(
        image
    ).all():

        raise RuntimeError(
            f"Non-finite training image "
            f"at index {idx}"
        )


    pixels = image.reshape(
        2,
        -1
    ).astype(
        np.float64,
        copy=False
    )


    local_sum += (
        pixels.sum(
            axis=1
        )
    )


    local_sum_sq += (
        np.square(
            pixels
        ).sum(
            axis=1
        )
    )


    local_count += (
        pixels.shape[1]
    )


# ------------------------------------------------------------
# Move statistics to CPU tensors for distributed reduction.
# ------------------------------------------------------------

sum_tensor = torch.tensor(
    local_sum,
    dtype=torch.float64,
    device=device
)

sum_sq_tensor = torch.tensor(
    local_sum_sq,
    dtype=torch.float64,
    device=device
)

count_tensor = torch.tensor(
    [local_count],
    dtype=torch.float64,
    device=device
)


dist.all_reduce(
    sum_tensor,
    op=dist.ReduceOp.SUM
)

dist.all_reduce(
    sum_sq_tensor,
    op=dist.ReduceOp.SUM
)

dist.all_reduce(
    count_tensor,
    op=dist.ReduceOp.SUM
)


total_count = (
    count_tensor.item()
)


mean_tensor = (
    sum_tensor
    /
    total_count
)


variance_tensor = (
    sum_sq_tensor
    /
    total_count
    -
    mean_tensor.pow(2)
)


variance_tensor = torch.clamp(
    variance_tensor,
    min=1e-8
)


std_tensor = torch.sqrt(
    variance_tensor
)


mean_values = (
    mean_tensor
    .cpu()
    .numpy()
    .astype(np.float32)
)


std_values = (
    std_tensor
    .cpu()
    .numpy()
    .astype(np.float32)
)


if rank == 0:

    print(
        f"VV mean : {mean_values[0]:.6f}"
    )

    print(
        f"VV std  : {std_values[0]:.6f}"
    )

    print(
        f"VH mean : {mean_values[1]:.6f}"
    )

    print(
        f"VH std  : {std_values[1]:.6f}"
    )

    print()


# ============================================================
# NORMALIZED DATASET
# ============================================================

class NormalizedPatchDataset(Dataset):

    def __init__(
        self,
        manifest,
        split,
        mean_values,
        std_values
    ):

        self.rows = []

        self.mean = (
            np.asarray(
                mean_values,
                dtype=np.float32
            )
            .reshape(
                2,
                1,
                1
            )
        )

        self.std = (
            np.asarray(
                std_values,
                dtype=np.float32
            )
            .reshape(
                2,
                1,
                1
            )
        )


        with open(
            manifest,
            "r"
        ) as f:

            header = (
                f.readline()
                .strip()
                .split(",")
            )

            for line in f:

                vals = (
                    line
                    .strip()
                    .split(",")
                )

                row = dict(
                    zip(
                        header,
                        vals
                    )
                )

                if row["split"] == split:

                    self.rows.append(
                        row
                    )


    def __len__(self):

        return len(
            self.rows
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.rows[idx]

        image = np.load(
            os.path.join(
                BASE,
                row["image"]
            ),
            mmap_mode="r"
        )

        mask = np.load(
            os.path.join(
                BASE,
                row["mask"]
            ),
            mmap_mode="r"
        )


        image = np.asarray(
            image,
            dtype=np.float32
        )

        mask = np.asarray(
            mask,
            dtype=np.float32
        )


        if not np.isfinite(
            image
        ).all():

            raise RuntimeError(
                f"Non-finite image: "
                f"{row['image']}"
            )


        image = (
            image - self.mean
        ) / self.std


        image = np.ascontiguousarray(
            image,
            dtype=np.float32
        )

        mask = np.ascontiguousarray(
            mask,
            dtype=np.float32
        )


        return (
            torch.from_numpy(
                image
            ),
            torch.from_numpy(
                mask
            ).unsqueeze(0)
        )


train_ds = NormalizedPatchDataset(
    manifest,
    "train",
    mean_values,
    std_values
)


val_ds = NormalizedPatchDataset(
    manifest,
    "val",
    mean_values,
    std_values
)


# ============================================================
# DISTRIBUTED LOADERS
# ============================================================

train_sampler = DistributedSampler(
    train_ds,
    num_replicas=world_size,
    rank=rank,
    shuffle=True,
    seed=SEED
)


val_sampler = DistributedSampler(
    val_ds,
    num_replicas=world_size,
    rank=rank,
    shuffle=False
)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_PER_GPU,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE_PER_GPU,
    sampler=val_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


# ============================================================
# U-NET
# Matches original checkpoint:
# Conv2d uses bias=False.
# ============================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_ch
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_ch
            ),

            nn.ReLU(
                inplace=True
            )
        )


    def forward(
        self,
        x
    ):

        return self.block(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(
            2,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.enc4 = DoubleConv(
            128,
            256
        )

        self.pool = nn.MaxPool2d(
            2
        )

        self.bottleneck = DoubleConv(
            256,
            512
        )

        self.up4 = nn.ConvTranspose2d(
            512,
            256,
            2,
            stride=2
        )

        self.dec4 = DoubleConv(
            512,
            256
        )

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.out = nn.Conv2d(
            32,
            1,
            1
        )


    def forward(
        self,
        x
    ):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        b = self.bottleneck(
            self.pool(e4)
        )

        d4 = self.up4(b)

        d4 = torch.cat(
            [d4, e4],
            dim=1
        )

        d4 = self.dec4(d4)

        d3 = self.up3(d4)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(d3)

        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(d2)

        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(d1)

        return self.out(d1)


# ============================================================
# MODEL
# ============================================================

model = UNet().to(
    device
)


model = DDP(
    model,
    device_ids=[local_rank],
    output_device=local_rank
)


# ============================================================
# LOSS
# ============================================================

bce = nn.BCEWithLogitsLoss()


def dice_loss(
    logits,
    targets
):

    logits = logits.float()

    targets = targets.float()

    probs = torch.sigmoid(
        logits
    )

    intersection = (
        probs * targets
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        probs.sum(
            dim=(1, 2, 3)
        )
        +
        targets.sum(
            dim=(1, 2, 3)
        )
    )

    dice = (
        2.0 * intersection + 1.0
    ) / (
        denominator + 1.0
    )

    return (
        1.0 - dice.mean()
    )


def combined_loss(
    logits,
    targets
):

    logits = logits.float()

    targets = targets.float()

    return (
        0.5 * bce(
            logits,
            targets
        )
        +
        0.5 * dice_loss(
            logits,
            targets
        )
    )


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS
)


# ============================================================
# LOAD EPOCH-7 MODEL
#
# ONLY MODEL WEIGHTS ARE LOADED.
# Optimizer and scheduler are intentionally fresh.
# ============================================================

if not os.path.exists(
    BEST_MODEL
):

    if rank == 0:

        print(
            "ERROR: best_model.pt not found."
        )

    dist.destroy_process_group()

    raise FileNotFoundError(
        BEST_MODEL
    )


checkpoint = torch.load(
    BEST_MODEL,
    map_location=device,
    weights_only=False
)


model.module.load_state_dict(
    checkpoint["model"]
)


best_dice = float(
    checkpoint["best_dice"]
)


best_epoch = int(
    checkpoint["best_epoch"]
)


old_history = checkpoint.get(
    "history",
    []
)


history = list(
    old_history
)


start_epoch = (
    best_epoch + 1
)


if rank == 0:

    print()
    print("=" * 70)
    print("VALID BEST MODEL LOADED")
    print("=" * 70)

    print(
        f"Checkpoint epoch : {best_epoch}"
    )

    print(
        f"Best Dice        : {best_dice:.6f}"
    )

    print(
        f"Starting epoch   : {start_epoch}"
    )

    print(
        f"Learning rate    : {LR}"
    )

    print(
        "Normalization    : Training-set VV/VH"
    )

    print(
        "Precision        : FP32"
    )

    print(
        "Optimizer        : Fresh AdamW"
    )

    print()


# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def validate():

    model.eval()

    dice_sum = torch.tensor(
        0.0,
        device=device
    )

    iou_sum = torch.tensor(
        0.0,
        device=device
    )

    count = torch.tensor(
        0.0,
        device=device
    )


    for images, masks in val_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )


        logits = model(
            images
        )


        logits = logits.float()


        if not torch.isfinite(
            logits
        ).all():

            raise RuntimeError(
                "Non-finite validation logits"
            )


        probs = torch.sigmoid(
            logits
        )


        preds = (
            probs >= 0.5
        ).float()


        intersection = (
            preds * masks
        ).sum(
            dim=(1, 2, 3)
        )


        pred_sum = preds.sum(
            dim=(1, 2, 3)
        )


        mask_sum = masks.sum(
            dim=(1, 2, 3)
        )


        union = (
            pred_sum
            +
            mask_sum
            -
            intersection
        )


        dice = (
            2.0 * intersection + 1.0
        ) / (
            pred_sum
            +
            mask_sum
            +
            1.0
        )


        iou = (
            intersection + 1.0
        ) / (
            union + 1.0
        )


        dice_sum += dice.sum()

        iou_sum += iou.sum()

        count += dice.numel()


    dist.all_reduce(
        dice_sum
    )

    dist.all_reduce(
        iou_sum
    )

    dist.all_reduce(
        count
    )


    return (
        (
            dice_sum / count
        ).item(),

        (
            iou_sum / count
        ).item()
    )


# ============================================================
# CHECKPOINT SAVE
# ============================================================

def save_checkpoint(
    path,
    epoch
):

    if rank != 0:
        return


    checkpoint = {

        "epoch":
            epoch,

        "model":
            model.module.state_dict(),

        "optimizer":
            optimizer.state_dict(),

        "scheduler":
            scheduler.state_dict(),

        "best_dice":
            best_dice,

        "best_epoch":
            best_epoch,

        "history":
            history,

        "normalization_mean":
            mean_values.tolist(),

        "normalization_std":
            std_values.tolist(),

        "learning_rate":
            LR
    }


    temporary = (
        path + ".tmp"
    )


    torch.save(
        checkpoint,
        temporary
    )


    os.replace(
        temporary,
        path
    )


# ============================================================
# TRAINING
# ============================================================

total_start = time.time()


for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1
):

    train_sampler.set_epoch(
        epoch
    )

    model.train()

    epoch_start = time.time()

    running_loss = 0.0

    samples_seen = 0

    bad = False


    # ========================================================
    # TRAINING BATCHES
    # ========================================================

    for step, (
        images,
        masks
    ) in enumerate(
        train_loader
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # INPUT CHECK
        # ----------------------------------------------------

        if not torch.isfinite(
            images
        ).all():

            bad = True

            if rank == 0:

                print(
                    f"ERROR: Non-finite input "
                    f"at epoch {epoch}, "
                    f"step {step}"
                )

            break


        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # FP32 FORWARD
        # ----------------------------------------------------

        logits = model(
            images
        )


        # ----------------------------------------------------
        # LOGIT CHECK
        # ----------------------------------------------------

        if not torch.isfinite(
            logits
        ).all():

            bad = True

            if rank == 0:

                print(
                    f"ERROR: Non-finite logits "
                    f"at epoch {epoch}, "
                    f"step {step}"
                )

            break


        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss = combined_loss(
            logits,
            masks
        )


        # ----------------------------------------------------
        # LOSS CHECK
        # ----------------------------------------------------

        if not torch.isfinite(
            loss
        ):

            bad = True

            if rank == 0:

                print(
                    f"ERROR: Non-finite loss "
                    f"at epoch {epoch}, "
                    f"step {step}"
                )

            break


        # ----------------------------------------------------
        # BACKWARD
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # CLIP GRADIENT
        # ----------------------------------------------------

        grad_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )
        )


        # ----------------------------------------------------
        # GRADIENT CHECK
        # ----------------------------------------------------

        if not torch.isfinite(
            grad_norm
        ):

            bad = True

            if rank == 0:

                print(
                    f"ERROR: Non-finite gradient "
                    f"at epoch {epoch}, "
                    f"step {step}"
                )

            optimizer.zero_grad(
                set_to_none=True
            )

            break


        # ----------------------------------------------------
        # OPTIMIZER
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # PARAMETER CHECK
        # ----------------------------------------------------

        parameters_ok = True


        for parameter in (
            model.module.parameters()
        ):

            if not torch.isfinite(
                parameter
            ).all():

                parameters_ok = False

                break


        if not parameters_ok:

            bad = True

            if rank == 0:

                print(
                    f"ERROR: Non-finite parameters "
                    f"at epoch {epoch}, "
                    f"step {step}"
                )

            break


        running_loss += (
            loss.item()
            *
            images.size(0)
        )


        samples_seen += (
            images.size(0)
        )


    # ========================================================
    # SYNCHRONIZE FAILURE STATUS
    # ========================================================

    bad_tensor = torch.tensor(
        1 if bad else 0,
        dtype=torch.int32,
        device=device
    )


    dist.all_reduce(
        bad_tensor,
        op=dist.ReduceOp.MAX
    )


    if bad_tensor.item() != 0:

        if rank == 0:

            print()
            print("=" * 70)
            print("TRAINING STOPPED SAFELY")
            print("=" * 70)
            print(
                "Non-finite value detected."
            )
            print(
                "No bad checkpoint was saved."
            )
            print(
                f"Best valid model remains:"
            )
            print(
                BEST_MODEL
            )
            print("=" * 70)


        dist.barrier()

        break


    # ========================================================
    # REDUCE TRAIN LOSS
    # ========================================================

    loss_tensor = torch.tensor(
        [
            running_loss,
            samples_seen
        ],
        dtype=torch.float64,
        device=device
    )


    dist.all_reduce(
        loss_tensor,
        op=dist.ReduceOp.SUM
    )


    train_loss = (
        loss_tensor[0]
        /
        loss_tensor[1]
    ).item()


    # ========================================================
    # LR
    # ========================================================

    scheduler.step()


    # ========================================================
    # VALIDATION
    # ========================================================

    val_dice, val_iou = (
        validate()
    )


    # ========================================================
    # TIME
    # ========================================================

    epoch_time = (
        time.time()
        -
        epoch_start
    )


    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    # ========================================================
    # HISTORY RECORD
    # ========================================================

    record = {

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_dice":
            val_dice,

        "val_iou":
            val_iou,

        "lr":
            current_lr,

        "epoch_time_sec":
            epoch_time
    }


    history.append(
        record
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    improved = (
        val_dice
        >
        best_dice + MIN_DELTA
    )


    if improved:

        best_dice = val_dice

        best_epoch = epoch

        save_checkpoint(
            BEST_MODEL,
            epoch
        )


    # ========================================================
    # EPOCH CHECKPOINT
    # ========================================================

    epoch_checkpoint = (
        f"{CKPT_DIR}/"
        f"checkpoint_epoch_{epoch:03d}.pt"
    )


    save_checkpoint(
        epoch_checkpoint,
        epoch
    )


    # ========================================================
    # LAST CHECKPOINT
    # ========================================================

    save_checkpoint(
        f"{CKPT_DIR}/last_checkpoint.pt",
        epoch
    )


    # ========================================================
    # HISTORY
    # ========================================================

    if rank == 0:

        with open(
            f"{WORK}/history.json",
            "w"
        ) as f:

            json.dump(
                history,
                f,
                indent=2
            )


        elapsed = (
            time.time()
            -
            total_start
        )


        epochs_done = (
            epoch
            -
            start_epoch
            +
            1
        )


        avg_epoch = (
            elapsed
            /
            max(
                1,
                epochs_done
            )
        )


        remaining = (
            MAX_EPOCHS
            -
            epoch
        )


        eta_hours = (
            avg_epoch
            *
            remaining
            /
            3600
        )


        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"loss {train_loss:.5f} | "
            f"Dice {val_dice:.5f} | "
            f"IoU {val_iou:.5f} | "
            f"LR {current_lr:.2e} | "
            f"{epoch_time/60:.2f} min | "
            f"ETA {eta_hours:.2f} h"
        )


        if improved:

            print(
                f"  NEW BEST — "
                f"Dice {best_dice:.5f}"
            )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epoch >= CHECK_EVERY
        and
        epoch % CHECK_EVERY == 0
    ):

        recent_start = (
            epoch
            -
            CHECK_EVERY
            +
            1
        )


        previous_start = (
            epoch
            -
            2 * CHECK_EVERY
            +
            1
        )


        recent = [

            x["val_dice"]

            for x in history

            if (
                recent_start
                <=
                x["epoch"]
                <=
                epoch
            )
        ]


        previous = [

            x["val_dice"]

            for x in history

            if (
                previous_start
                <=
                x["epoch"]
                <
                recent_start
            )
        ]


        if (
            recent
            and
            previous
        ):

            recent_best = max(
                recent
            )

            previous_best = max(
                previous
            )


            if (
                recent_best
                <=
                previous_best
                +
                MIN_DELTA
            ):

                if rank == 0:

                    print()
                    print(
                        "Early stopping condition reached."
                    )

                break


# ============================================================
# FINAL SUMMARY
# ============================================================

if rank == 0:

    print()
    print("=" * 70)
    print("TRAINING FINISHED")
    print("=" * 70)

    print(
        f"Best Dice : {best_dice:.6f}"
    )

    print(
        f"Best Epoch: {best_epoch}"
    )

    print()

    print(
        "Best model:"
    )

    print(
        BEST_MODEL
    )

    print()

    print(
        "History:"
    )

    print(
        f"{WORK}/history.json"
    )

    print()

    print(
        "Epoch checkpoints:"
    )

    print(
        f"{CKPT_DIR}/checkpoint_epoch_XXX.pt"
    )


dist.destroy_process_group()
'''

script_path = Path(
    f"{WORK}/train_robust_ddp.py"
)

script_path.write_text(
    script
)

print("=" * 70)
print("FINAL ROBUST TRAINING SCRIPT CREATED")
print("=" * 70)
print()
print(script_path)
print()
print("Starting model : Epoch-7 best_model.pt")
print("Starting epoch : 8")
print("Learning rate  : 1e-4")
print("Normalization  : training-set statistics")
print("Precision      : FP32")
print("Gradient clip  : 1.0") 
print("GPUs           : 2")
print("Batch/GPU      : 4")

FINAL ROBUST TRAINING SCRIPT CREATED

/kaggle/working/sar_oil_spill_robust/train_robust_ddp.py

Starting model : Epoch-7 best_model.pt
Starting epoch : 8
Learning rate  : 1e-4
Normalization  : training-set statistics
Precision      : FP32
Gradient clip  : 1.0
GPUs           : 2
Batch/GPU      : 4


In [1]:
# ============================================================
# CELL 2 — LAUNCH
# ============================================================

import os
import subprocess

work = "/kaggle/working/sar_oil_spill_robust"

script = (
    f"{work}/train_robust_ddp.py"
)

env = os.environ.copy()

env["MASTER_ADDR"] = "127.0.0.1"

env["MASTER_PORT"] = "29506"


print("=" * 70)
print("STARTING FINAL ROBUST 2-GPU TRAINING")
print("=" * 70)
print()
print("Starting model : Epoch-7 best_model.pt")
print("Starting epoch : 8")
print("Learning rate  : 1e-4")
print("Normalization  : training-set VV/VH statistics")
print("Precision      : FP32")
print("Gradient clip  : 1.0")
print("GPUs           : 2")
print("Batch/GPU      : 4")
print()

result = subprocess.run(
    [
        "torchrun",
        "--standalone",
        "--nproc_per_node=2",
        script
    ],
    env=env
)

print()
print("=" * 70)
print(
    "Training process exited with code:",
    result.returncode
)
print("=" * 70)

STARTING FINAL ROBUST 2-GPU TRAINING

Starting model : Epoch-7 best_model.pt
Starting epoch : 8
Learning rate  : 1e-4
Normalization  : training-set VV/VH statistics
Precision      : FP32
Gradient clip  : 1.0
GPUs           : 2
Batch/GPU      : 4



W0917 05:20:05.546000 106 torch/distributed/run.py:852] 
W0917 05:20:05.546000 106 torch/distributed/run.py:852] *****************************************
W0917 05:20:05.546000 106 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0917 05:20:05.546000 106 torch/distributed/run.py:852] *****************************************
[W917 05:20:05.783691541 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W917 05:20:08.025233359 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W917 05:20:08.028370239 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3



CALCULATING TRAINING-SET NORMALIZATION


W0917 05:20:16.571000 106 torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W0917 05:20:16.573000 106 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 114 closing signal SIGINT
W0917 05:20:16.574000 106 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 115 closing signal SIGINT
[rank0]: Traceback (most recent call last):
[rank0]:   File "/kaggle/working/sar_oil_spill_robust/train_robust_ddp.py", line 261, in <module>
[rank0]:     image, _ = raw_train_ds[idx]
[rank0]:                ~~~~~~~~~~~~^^^^^
[rank0]:   File "/kaggle/working/sar_oil_spill_robust/train_robust_ddp.py", line 180, in __getitem__
[rank0]:     mask = np.load(
[rank0]:            ^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 455, in load
[rank0]:     fid = stack.enter_context(open(os.fspath(file), "rb"))
[rank0]:                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]:

KeyboardInterrupt: 

In [3]:
!find /kaggle/working -type f \( -name "*.pth" -o -name "*.pt" -o -name "*.ckpt" \) -ls

   131120  91096 -rw-r--r--   1 root     root     93278547 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_022.pt
   131113  91096 -rw-r--r--   1 root     root     93279123 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_029.pt
   131099  91096 -rw-r--r--   1 root     root     93278611 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_023.pt
   131109  91092 -rw-r--r--   1 root     root     93277971 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_015.pt
   131100  91096 -rw-r--r--   1 root     root     93279187 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_030.pt
   131112  91092 -rw-r--r--   1 root     root     93278099 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_017.pt
   131117  91092 -rw-r--r--   1 root     root     93277907 Sep 17 05:06 /kaggle/working/sar_oil_spill_robust/checkpoints/checkpoin

In [5]:
import torch

CKPT = "/kaggle/working/sar_oil_spill_robust/checkpoints/last_checkpoint.pt"

ckpt = torch.load(CKPT, map_location="cpu")

print("Checkpoint keys:")
print(list(ckpt.keys()))

print("\nNon-dictionary values:")
for k, v in ckpt.items():
    if not isinstance(v, dict):
        print(f"{k}: {v}") 


Checkpoint keys:
['epoch', 'model', 'optimizer', 'scheduler', 'best_dice', 'best_epoch', 'history', 'normalization_mean', 'normalization_std', 'learning_rate']

Non-dictionary values:
epoch: 30
best_dice: 0.8298674821853638
best_epoch: 19
history: [{'epoch': 1, 'train_loss': 0.39851417584092513, 'val_dice': 0.7683228850364685, 'val_iou': 0.7396451234817505, 'lr': 0.0009997535269026829, 'epoch_time_sec': 375.4121525287628, 'best_dice': 0.7683228850364685}, {'epoch': 2, 'train_loss': 0.17586741870359218, 'val_dice': 0.8161144256591797, 'val_iou': 0.7806596755981445, 'lr': 0.0009990143508499217, 'epoch_time_sec': 368.30394983291626, 'best_dice': 0.8161144256591797}, {'epoch': 3, 'train_loss': 0.15118483684473555, 'val_dice': 0.8125811219215393, 'val_iou': 0.7771651148796082, 'lr': 0.0009977832013192385, 'epoch_time_sec': 366.4022741317749, 'best_dice': 0.8161144256591797}, {'epoch': 4, 'train_loss': 0.14350527333511043, 'val_dice': 0.7993969321250916, 'val_iou': 0.7675554752349854, 'lr': 

In [10]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [11]:
import os
import torch

CKPT_PATH = "/kaggle/working/sar_oil_spill_robust/checkpoints/last_checkpoint.pt"

print("Checkpoint exists:", os.path.exists(CKPT_PATH))
print("Checkpoint size:",
      f"{os.path.getsize(CKPT_PATH) / 1024**2:.2f} MB")

checkpoint = torch.load(CKPT_PATH, map_location="cpu")

print("\nCheckpoint contents:")
print(list(checkpoint.keys()))

print("\nTraining state:")
print("Last epoch :", checkpoint["epoch"])
print("Best Dice  :", checkpoint["best_dice"])
print("Best epoch :", checkpoint["best_epoch"])
print("LR         :", checkpoint["learning_rate"])

Checkpoint exists: True
Checkpoint size: 88.96 MB

Checkpoint contents:
['epoch', 'model', 'optimizer', 'scheduler', 'best_dice', 'best_epoch', 'history', 'normalization_mean', 'normalization_std', 'learning_rate']

Training state:
Last epoch : 30
Best Dice  : 0.8298674821853638
Best epoch : 19
LR         : 0.0001


In [22]:
# ============================================================
# EXACT CHECKPOINT-COMPATIBLE U-NET
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetCheckpoint(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        # EXACT names expected by checkpoint
        self.enc1 = ConvBlock(2, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.enc4 = ConvBlock(128, 256)

        self.bottleneck = ConvBlock(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        # IMPORTANT: checkpoint expects "out", not "final"
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        b = self.bottleneck(
            self.pool(e4)
        )

        x = self.up4(b)
        x = self.dec4(
            torch.cat([x, e4], dim=1)
        )

        x = self.up3(x)
        x = self.dec3(
            torch.cat([x, e3], dim=1)
        )

        x = self.up2(x)
        x = self.dec2(
            torch.cat([x, e2], dim=1)
        )

        x = self.up1(x)
        x = self.dec1(
            torch.cat([x, e1], dim=1)
        )

        return self.out(x)


# Create model
model = UNetCheckpoint().cuda()

# Verify checkpoint compatibility
ckpt_keys = list(checkpoint["model"].keys())
model_keys = list(model.state_dict().keys())

print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Checkpoint keys:", len(ckpt_keys))
print("Model keys:", len(model_keys))
print("Exact key match:", ckpt_keys == model_keys)

if ckpt_keys != model_keys:
    print("\nCheckpoint only:")
    print([k for k in ckpt_keys if k not in model_keys])

    print("\nModel only:")
    print([k for k in model_keys if k not in ckpt_keys])
else:
    print("✓ EXACT CHECKPOINT MATCH")

Parameters: 7,762,753
Checkpoint keys: 118
Model keys: 118
Exact key match: True
✓ EXACT CHECKPOINT MATCH


In [21]:
# ============================================================
# STEP 4 — FIX FINAL LAYER NAME
# ============================================================

model.final = model.out if hasattr(model, "out") else model.final

In [23]:
# ============================================================
# STEP 5 — RESTORE COMPLETE TRAINING STATE
# ============================================================

CKPT_PATH = "/kaggle/working/sar_oil_spill_robust/checkpoints/last_checkpoint.pt"

# Load checkpoint
checkpoint = torch.load(
    CKPT_PATH,
    map_location="cuda"
)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100,
    eta_min=1e-6
)

# ------------------------------------------------------------
# Restore saved states
# ------------------------------------------------------------

model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])
scheduler.load_state_dict(checkpoint["scheduler"])

# ------------------------------------------------------------
# Restore training information
# ------------------------------------------------------------

start_epoch = checkpoint["epoch"] + 1
best_dice = checkpoint["best_dice"]
best_epoch = checkpoint["best_epoch"]
history = checkpoint["history"]

normalization_mean = checkpoint["normalization_mean"]
normalization_std = checkpoint["normalization_std"]

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING STATE RESTORED")
print("=" * 70)

print(f"Last completed epoch : {checkpoint['epoch']}")
print(f"Next epoch           : {start_epoch}")
print(f"Best Dice            : {best_dice:.6f}")
print(f"Best epoch           : {best_epoch}")
print(f"Current LR           : {optimizer.param_groups[0]['lr']:.10f}")

print("\nNormalization:")
print("Mean:", normalization_mean)
print("Std :", normalization_std)

print("\n✓ Model weights restored")
print("✓ Optimizer restored")
print("✓ Scheduler restored")
print("✓ Training history restored")

print("=" * 70)

TRAINING STATE RESTORED
Last completed epoch : 30
Next epoch           : 31
Best Dice            : 0.829867
Best epoch           : 19
Current LR           : 0.0000875056

Normalization:
Mean: [-33.32752990722656, -19.957395553588867]
Std : [5.795812606811523, 4.1462554931640625]

✓ Model weights restored
✓ Optimizer restored
✓ Scheduler restored
✓ Training history restored


In [24]:
print("train_loader:", "OK" if "train_loader" in globals() else "MISSING")
print("val_loader:", "OK" if "val_loader" in globals() else "MISSING")
print("loss_fn:", "OK" if "loss_fn" in globals() else "MISSING")
print("scheduler:", "OK" if "scheduler" in globals() else "MISSING")
print("optimizer:", "OK" if "optimizer" in globals() else "MISSING")

train_loader: MISSING
val_loader: MISSING
loss_fn: MISSING
scheduler: OK
optimizer: OK


In [26]:
from pathlib import Path

print("Searching /kaggle/input for part1_manifest.csv ...")

matches = list(
    Path("/kaggle/input").rglob("part1_manifest.csv")
)

for p in matches:
    print("FOUND:", p)

if not matches:
    print("\n❌ part1_manifest.csv was not found.")
    print("\nAvailable datasets:")
    for p in Path("/kaggle/input").iterdir():
        print(" ", p)
else:
    print("\n✓ Dataset found")

Searching /kaggle/input for part1_manifest.csv ...
FOUND: /kaggle/input/datasets/vssrinivas001/sentinel1-sar-oil-spill-part1/metadata/part1_manifest.csv

✓ Dataset found


In [27]:
# ============================================================
# STEP 6 — RECREATE DATASET + LOADERS + LOSS + METRICS
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Correct Kaggle dataset path
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/vssrinivas001/"
    "sentinel1-sar-oil-spill-part1"
)

BATCH_SIZE = 4
NUM_WORKERS = 2

# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

manifest = pd.read_csv(
    DATA_ROOT / "metadata" / "part1_manifest.csv"
)

train_df = manifest[
    manifest["split"] == "train"
].reset_index(drop=True)

val_df = manifest[
    manifest["split"] == "val"
].reset_index(drop=True)

print("Manifest loaded")
print(f"Train patches: {len(train_df):,}")
print(f"Val patches  : {len(val_df):,}")

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class SARDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = np.load(
            DATA_ROOT / row["image"]
        ).astype(np.float32)

        mask = np.load(
            DATA_ROOT / row["mask"]
        ).astype(np.float32)

        image = torch.from_numpy(
            image.copy()
        )

        mask = torch.from_numpy(
            mask.copy()
        ).unsqueeze(0)

        mask = (mask > 0).float()

        return image, mask


train_ds = SARDataset(train_df)
val_ds = SARDataset(val_df)

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

def loss_fn(logits, target):

    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        target
    )

    prob = torch.sigmoid(logits)

    intersection = (
        prob * target
    ).sum(dim=(1, 2, 3))

    dice = (
        2 * intersection + 1
    ) / (
        prob.sum(dim=(1, 2, 3))
        + target.sum(dim=(1, 2, 3))
        + 1
    )

    return bce + (1 - dice.mean())


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

@torch.no_grad()
def get_metrics(logits, target):

    pred = (
        torch.sigmoid(logits) > 0.5
    ).float()

    intersection = (
        pred * target
    ).sum()

    dice = (
        2 * intersection + 1
    ) / (
        pred.sum()
        + target.sum()
        + 1
    )

    union = (
        pred.sum()
        + target.sum()
        - intersection
    )

    iou = (
        intersection + 1
    ) / (union + 1)

    return dice, iou


print()
print("=" * 70)
print("DATA PIPELINE READY")
print("=" * 70)
print(f"Train patches : {len(train_ds):,}")
print(f"Val patches   : {len(val_ds):,}")
print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Batch size    : {BATCH_SIZE}")
print("=" * 70)

# ------------------------------------------------------------
# Quick sample test
# ------------------------------------------------------------

x, y = train_ds[0]

print()
print("Sample check:")
print("Image shape:", x.shape)
print("Image dtype:", x.dtype)
print("Mask shape :", y.shape)
print("Mask dtype :", y.dtype)

Manifest loaded
Train patches: 15,360
Val patches  : 3,840

DATA PIPELINE READY
Train patches : 15,360
Val patches   : 3,840
Train batches : 3,840
Val batches   : 960
Batch size    : 4

Sample check:
Image shape: torch.Size([2, 512, 512])
Image dtype: torch.float32
Mask shape : torch.Size([1, 512, 512])
Mask dtype : torch.float32


In [28]:
# ============================================================
# RESUME TRAINING: EPOCH 31 -> 100
# ============================================================

import os
import time
import torch

NUM_EPOCHS = 100

CHECKPOINT_DIR = "/kaggle/working/sar_oil_spill_robust/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Verify required objects
# ------------------------------------------------------------

required = [
    "model",
    "optimizer",
    "scheduler",
    "train_loader",
    "val_loader",
    "loss_fn",
    "get_metrics",
    "start_epoch",
    "best_dice",
    "best_epoch",
    "history",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(f"Missing objects: {missing}")

print("=" * 70)
print("RESUMING TRAINING")
print("=" * 70)
print(f"Starting epoch : {start_epoch}")
print(f"Ending epoch   : {NUM_EPOCHS}")
print(f"Best Dice      : {best_dice:.6f}")
print(f"Best epoch     : {best_epoch}")
print(f"Current LR     : {optimizer.param_groups[0]['lr']:.10f}")
print("=" * 70)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    running_loss = 0.0
    total_batches = len(train_loader)

    print(f"\nEpoch {epoch:03d}/{NUM_EPOCHS}")
    print("-" * 70)

    for batch_idx, (x, y) in enumerate(train_loader):

        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            pred = model(x)
            loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Live progress
        done = batch_idx + 1

        if done % 20 == 0 or done == total_batches:

            elapsed = time.time() - epoch_start
            rate = done / max(elapsed, 0.001)
            remaining = total_batches - done
            eta = remaining / max(rate, 1e-9)

            pct = done / total_batches * 100

            print(
                f"\rTrain [{pct:6.2f}%] "
                f"{done}/{total_batches} | "
                f"Loss {loss.item():.4f} | "
                f"ETA {eta/60:.1f} min",
                end="",
                flush=True
            )

    print()

    train_loss = running_loss / total_batches

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    dice_sum = 0.0
    iou_sum = 0.0
    sample_count = 0

    val_batches = len(val_loader)

    with torch.no_grad():

        for batch_idx, (x, y) in enumerate(val_loader):

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                pred = model(x)

            dice, iou = get_metrics(pred, y)

            n = x.size(0)

            dice_sum += dice.item() * n
            iou_sum += iou.item() * n
            sample_count += n

            if (batch_idx + 1) % 20 == 0 or \
               (batch_idx + 1) == val_batches:

                pct = (batch_idx + 1) / val_batches * 100

                print(
                    f"\rVal   [{pct:6.2f}%] "
                    f"{batch_idx + 1}/{val_batches}",
                    end="",
                    flush=True
                )

    print()

    val_dice = dice_sum / sample_count
    val_iou = iou_sum / sample_count

    # ========================================================
    # SCHEDULER
    # ========================================================

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = time.time() - epoch_start

    # ========================================================
    # BEST MODEL
    # ========================================================

    is_best = val_dice > best_dice

    if is_best:
        best_dice = val_dice
        best_epoch = epoch

    # ========================================================
    # HISTORY
    # ========================================================

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
        "lr": current_lr,
        "epoch_time_sec": epoch_time,
        "best_dice": best_dice,
    })

    # ========================================================
    # SAVE LAST CHECKPOINT
    # ========================================================

    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_dice": best_dice,
        "best_epoch": best_epoch,
        "history": history,
        "normalization_mean": normalization_mean,
        "normalization_std": normalization_std,
        "learning_rate": 1e-4,
    }

    torch.save(
        state,
        f"{CHECKPOINT_DIR}/last_checkpoint.pt"
    )

    torch.save(
        state,
        f"{CHECKPOINT_DIR}/checkpoint_epoch_{epoch:03d}.pt"
    )

    # --------------------------------------------------------
    # Save best model separately
    # --------------------------------------------------------

    if is_best:

        torch.save(
            state,
            f"{CHECKPOINT_DIR}/best_model.pt"
        )

    # ========================================================
    # REPORT
    # ========================================================

    print()
    print("=" * 70)
    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} COMPLETE"
    )
    print("=" * 70)
    print(f"Train Loss : {train_loss:.6f}")
    print(f"Val Dice   : {val_dice:.6f}")
    print(f"Val IoU    : {val_iou:.6f}")
    print(f"LR         : {current_lr:.10f}")
    print(f"Time       : {epoch_time/60:.2f} min")
    print(f"Best Dice  : {best_dice:.6f} (epoch {best_epoch})")

    if is_best:
        print("✓ NEW BEST MODEL")

    print("=" * 70)

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(f"Best Dice  : {best_dice:.6f}")
print(f"Best epoch : {best_epoch}")
print("=" * 70)

RESUMING TRAINING
Starting epoch : 31
Ending epoch   : 100
Best Dice      : 0.829867
Best epoch     : 19
Current LR     : 0.0000875056

Epoch 031/100
----------------------------------------------------------------------
Train [100.00%] 3840/3840 | Loss 0.3952 | ETA 0.0 min
Val   [100.00%] 960/960

Epoch 031/100 COMPLETE
Train Loss : 0.221425
Val Dice   : 0.459225
Val IoU    : 0.410660
LR         : 0.0000864484
Time       : 10.54 min
Best Dice  : 0.829867 (epoch 19)

Epoch 032/100
----------------------------------------------------------------------
Train [100.00%] 3840/3840 | Loss 0.3078 | ETA 0.0 min
Val   [100.00%] 960/960

Epoch 032/100 COMPLETE
Train Loss : 0.240094
Val Dice   : 0.692252
Val IoU    : 0.619407
LR         : 0.0000853553
Time       : 10.45 min
Best Dice  : 0.829867 (epoch 19)

Epoch 033/100
----------------------------------------------------------------------
Train [100.00%] 3840/3840 | Loss 0.0044 | ETA 0.0 min
Val   [100.00%] 960/960

Epoch 033/100 COMPLETE
Train

KeyboardInterrupt: 

In [29]:
from pathlib import Path

for p in Path("/kaggle/working").rglob("*"):
    if p.is_file() and ("checkpoint" in p.name.lower() or "epoch" in p.name.lower()):
        print(p)

/kaggle/working/best_model_epoch7.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_039.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_037.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_022.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_029.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_023.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_015.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_030.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_031.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_017.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_035.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_036.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_044.pt
/kaggle/working/sar_oil_spill_robust/checkpoints/checkpoint_epoch_042.pt
/kaggle/workin

In [30]:
from pathlib import Path
import shutil

ckpt_dir = Path("/kaggle/working/sar_oil_spill_robust/checkpoints")

# Preserve epoch 19 under an obvious name
epoch19 = ckpt_dir / "checkpoint_epoch_019.pt"
fallback = ckpt_dir / "best_model_epoch19.pt"

if epoch19.exists():
    shutil.copy2(epoch19, fallback)
    print(f"✓ Preserved fallback: {fallback}")
else:
    raise FileNotFoundError(f"Epoch 19 checkpoint not found: {epoch19}")

# Delete epochs 31–45
deleted = []

for epoch in range(31, 46):
    p = ckpt_dir / f"checkpoint_epoch_{epoch:03d}.pt"
    if p.exists():
        p.unlink()
        deleted.append(p.name)

# The 'last' checkpoint belongs to the interrupted/NaN continuation
last_ckpt = ckpt_dir / "last_checkpoint.pt"
if last_ckpt.exists():
    last_ckpt.unlink()
    deleted.append(last_ckpt.name)

print("\nDeleted:")
for x in deleted:
    print(" ", x)

print("\n✓ Epoch 19 fallback is preserved.")

✓ Preserved fallback: /kaggle/working/sar_oil_spill_robust/checkpoints/best_model_epoch19.pt

Deleted:
  checkpoint_epoch_031.pt
  checkpoint_epoch_032.pt
  checkpoint_epoch_033.pt
  checkpoint_epoch_034.pt
  checkpoint_epoch_035.pt
  checkpoint_epoch_036.pt
  checkpoint_epoch_037.pt
  checkpoint_epoch_038.pt
  checkpoint_epoch_039.pt
  checkpoint_epoch_040.pt
  checkpoint_epoch_041.pt
  checkpoint_epoch_042.pt
  checkpoint_epoch_043.pt
  checkpoint_epoch_044.pt
  checkpoint_epoch_045.pt
  last_checkpoint.pt

✓ Epoch 19 fallback is preserved.
